# Fase 8 — Fine-tuning neural no Colab (XLM-R · BERTimbau · BERTweet)

**Notebook autossuficiente.** Não precisa clonar repo nem instalar o pacote `hsc`:
o código de treino e de métricas está embutido aqui (idêntico ao do pipeline local, para
a comparação clássico-vs-neural ser justa e o resultado entrar direto no leaderboard).

### Como usar
1. **Runtime → Change runtime type → GPU (T4)**.
2. Rode as células em ordem. Na célula **3** você sobe os dois arquivos do corpus
   (`corpus_strict.parquet` e `corpus_broad.parquet`, que estão em `data/processed/` no seu PC).
3. No fim, baixe `hsc_neural_results.zip` e, no repo local, rode:
   `python notebooks/merge_neural_results.py hsc_neural_results.zip` → `hsc report` / `analyze` / `bias`.

O corpus já vem com o **UTF-8 do português corrigido**; a célula 3 tem uma trava que recusa
corpus com mojibake latino-1.


## 1 · GPU


In [ ]:
!nvidia-smi -L


## 2 · Dependências (HF stack por cima do torch do Colab)


In [ ]:
!pip install -q 'transformers>=4.40' 'datasets>=2.18' 'accelerate>=0.29' 'evaluate>=0.4' 'sentencepiece>=0.2'


## 3 · Subir o corpus (do seu PC) + trava anti-mojibake
Escolha `corpus_strict.parquet` e `corpus_broad.parquet` (pasta `data/processed/`).


In [ ]:
import pandas as pd, os
from google.colab import files
up = files.upload()   # selecione corpus_strict.parquet E corpus_broad.parquet

CORPUS = {}
for name in up:
    pol = 'strict' if 'strict' in name else 'broad' if 'broad' in name else None
    if pol:
        CORPUS[pol] = pd.read_parquet(name)

assert 'strict' in CORPUS and 'broad' in CORPUS, 'suba os DOIS parquets (strict e broad)'
for pol, c in CORPUS.items():
    assert 'split' in c.columns, f'{pol}: corpus sem coluna split (rode hsc split antes)'
    pt = ' '.join(c[c.source_dataset == 'pt_fortuna'].text_clean.head(2000))
    assert ('é' in pt) or ('ã' in pt), f'{pol}: PT sem acentos reais'
    assert 'Ã©' not in pt, f'{pol}: mojibake latin-1 (Ã©) no PT!'
    print(pol, '->', c.split.value_counts().to_dict())
print('corpus OK (UTF-8 verificado)')


## 4 · Métricas + predições (idêntico ao `hsc`, para o schema bater)


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)
POS = 1

def classification_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    out = {"n": int(len(y_true)), "n_hate": int((y_true == POS).sum()),
           "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
           "accuracy": float((y_true == y_pred).mean()),
           "precision_hate": float(p[1]), "recall_hate": float(r[1]), "f1_hate": float(f[1]),
           "precision_nothate": float(p[0]), "recall_nothate": float(r[0])}
    if y_score is not None and len(np.unique(y_true)) == 2:
        y_score = np.asarray(y_score)
        out["roc_auc"] = float(roc_auc_score(y_true, y_score))
        out["pr_auc"] = float(average_precision_score(y_true, y_score))
    return out

def best_threshold(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score, dtype=float)
    cands = np.unique(np.quantile(y_score, np.linspace(0.02, 0.98, 97)))
    best_t, best_f = 0.5, -1.0
    for t in cands:
        f = f1_score(y_true, (y_score >= t).astype(int), average="macro", zero_division=0)
        if f > best_f: best_f, best_t = f, float(t)
    return best_t

def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=1000, seed=42, alpha=0.05):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed); n = len(y_true); stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[b] = f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0)
    return float(np.percentile(stats, 100*alpha/2)), float(np.percentile(stats, 100*(1-alpha/2)))

def confusion(y_true, y_pred):
    return confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()

def breakdown(df, by):
    rows = []
    for key, g in df.groupby(by):
        m = classification_metrics(g["label"], g["pred"])
        rows.append({by: key, "n": m["n"], "n_hate": m["n_hate"],
                     "macro_f1": round(m["macro_f1"], 4), "recall_hate": round(m["recall_hate"], 4)})
    return sorted(rows, key=lambda r: str(r[by]))

def metrics_block(part, y_pred, y_score, seed):
    y = part["label"].values; y_pred = np.asarray(y_pred)
    m = classification_metrics(y, y_pred, y_score)
    lo, hi = bootstrap_macro_f1_ci(y, y_pred, seed=seed)
    m["macro_f1_ci95"] = [round(lo, 4), round(hi, 4)]; m["confusion"] = confusion(y, y_pred)
    pdf = part[["language", "source_dataset", "label"]].copy(); pdf["pred"] = y_pred
    m["by_language"] = breakdown(pdf, "language"); m["by_source"] = breakdown(pdf, "source_dataset")
    return m

def prediction_frame(part, y_score, threshold):
    y_score = np.asarray(y_score, dtype=float)
    return pd.DataFrame({"id": part["id"].values, "language": part["language"].values,
                         "source_dataset": part["source_dataset"].values,
                         "y_true": part["label"].values.astype(int), "y_score": y_score,
                         "y_pred": (y_score >= threshold).astype(int)})
print("helpers definidos")


## 5 · Configs dos 3 transformers (inline)


In [ ]:
CONFIGS = {
  'xlmr_multilingual': {
    'ckpt': 'FacebookAI/xlm-roberta-base', 'languages': None, 'max_length': 160,
    'epochs': 4, 'batch_size': 16, 'lr': 2e-5},
  'bertimbau_pt': {
    'ckpt': 'neuralmind/bert-base-portuguese-cased', 'languages': ['pt'], 'max_length': 160,
    'epochs': 4, 'batch_size': 16, 'lr': 2e-5},
  'bertweet_en': {
    'ckpt': 'vinai/bertweet-base', 'languages': ['en'], 'max_length': 128,
    'epochs': 4, 'batch_size': 16, 'lr': 2e-5},
}
TEXT_COL = 'text_clean'
print(list(CONFIGS))


## 6 · Função de treino (WeightedTrainer + limiar na validação)


In [ ]:
import os, json, torch
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

os.makedirs('reports/metrics', exist_ok=True)
os.makedirs('reports/predictions', exist_ok=True)
REGISTRY = {}

def train_one(name, policy, seed):
    cfg = CONFIGS[name]; torch.manual_seed(seed); np.random.seed(seed)
    df = CORPUS[policy]
    if cfg['languages']: df = df[df['language'].isin(cfg['languages'])]
    tr, va, te = (df[df.split == s] for s in ('train', 'val', 'test'))
    tok = AutoTokenizer.from_pretrained(cfg['ckpt'])
    def enc(b): return tok(b[TEXT_COL], truncation=True, max_length=cfg['max_length'])
    def to_ds(p):
        d = Dataset.from_pandas(p[[TEXT_COL, 'label']].reset_index(drop=True))
        return d.map(enc, batched=True).rename_column('label', 'labels')
    ds = {k: to_ds(p) for k, p in (('val', va), ('test', te))}; ds_tr = to_ds(tr)
    model = AutoModelForSequenceClassification.from_pretrained(cfg['ckpt'], num_labels=2)
    cnt = tr['label'].value_counts().to_dict(); n = len(tr)
    w = torch.tensor([n/(2*cnt.get(0,1)), n/(2*cnt.get(1,1))], dtype=torch.float)
    class WT(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            lab = inputs.pop('labels'); out = model(**inputs)
            loss = torch.nn.CrossEntropyLoss(weight=w.to(out.logits.device))(out.logits, lab)
            return (loss, out) if return_outputs else loss
    mid = f'{name}_{policy}_s{seed}'
    args = TrainingArguments(output_dir=f'/content/hf/{mid}', num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'], per_device_eval_batch_size=32,
        learning_rate=cfg['lr'], weight_decay=0.01, warmup_ratio=0.1, fp16=True,
        eval_strategy='epoch', save_strategy='no', logging_steps=50, seed=seed, report_to=[])
    tr_obj = WT(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds['val'],
                data_collator=DataCollatorWithPadding(tok))
    tr_obj.train()
    def sc(d): return torch.softmax(torch.tensor(tr_obj.predict(d).predictions), 1).numpy()[:, 1]
    val_s = sc(ds['val']); thr = best_threshold(va['label'].values, val_s)
    res = {'model_id': mid, 'config': name, 'family': 'neural', 'policy': policy, 'seed': seed,
           'hf_checkpoint': cfg['ckpt'], 'n_train': int(len(tr)),
           'threshold': round(float(thr), 4), 'splits': {}}
    for split, part, sco in (('val', va, val_s), ('test', te, sc(ds['test']))):
        yp = (np.asarray(sco) >= thr).astype(int)
        res['splits'][split] = metrics_block(part, yp, sco, seed)
        prediction_frame(part, sco, thr).to_parquet(f'reports/predictions/{mid}_{split}.parquet', index=False)
    json.dump(res, open(f'reports/metrics/{mid}.json', 'w'), indent=2)
    REGISTRY[mid] = {'path': f'models/{mid}/hf', 'config': name, 'family': 'neural',
        'policy': policy, 'seed': seed, 'hf_checkpoint': cfg['ckpt'], 'threshold': round(float(thr),4),
        'val_macro_f1': res['splits']['val']['macro_f1'], 'test_macro_f1': res['splits']['test']['macro_f1']}
    print(f"  {mid}: test macro-F1={res['splits']['test']['macro_f1']:.4f} "
          f"recall_hate={res['splits']['test']['recall_hate']:.4f}")
    return res
print("train_one pronto")


## 7 · Treinar (resumível)
Comece com `SEEDS = [42]` para ter a comparação rápido; troque para `[42, 43, 44]` para o
intervalo de confiança do artigo. Se a sessão cair, rode de novo — pula o que já terminou.


In [ ]:
POLICIES = ['strict', 'broad']
SEEDS = [42]     # -> [42, 43, 44] para IC

for name in CONFIGS:
    for pol in POLICIES:
        for sd in SEEDS:
            mid = f'{name}_{pol}_s{sd}'
            if os.path.exists(f'reports/metrics/{mid}.json'):
                # recarrega no registry mesmo se já treinado
                r = json.load(open(f'reports/metrics/{mid}.json'))
                REGISTRY[mid] = {'path': f'models/{mid}/hf', 'config': name, 'family': 'neural',
                    'policy': pol, 'seed': sd, 'hf_checkpoint': CONFIGS[name]['ckpt'],
                    'threshold': r['threshold'], 'val_macro_f1': r['splits']['val']['macro_f1'],
                    'test_macro_f1': r['splits']['test']['macro_f1']}
                print('skip (ja feito):', mid); continue
            print('==>', mid, flush=True)
            train_one(name, pol, sd)
print('treino concluido')


## 8 · Baixar resultados (para o merge local)


In [ ]:
import zipfile
json.dump(REGISTRY, open('registry_neural.json', 'w'), indent=2)
with zipfile.ZipFile('hsc_neural_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('reports/metrics'):
        if f.endswith('.json'): z.write(f'reports/metrics/{f}', f'reports/metrics/{f}')
    for f in os.listdir('reports/predictions'):
        if f.endswith('.parquet'): z.write(f'reports/predictions/{f}', f'reports/predictions/{f}')
    z.write('registry_neural.json', 'models/registry_neural.json')
from google.colab import files
files.download('hsc_neural_results.zip')
print('pronto — no repo local: python notebooks/merge_neural_results.py hsc_neural_results.zip')
